# Proyecto PYE - Analisis UYU/BRL

Notebook preparado para Google Colab. Reproduce la carga, validacion, analisis descriptivo, graficos, autocorrelacion, bondad de ajuste chi-cuadrado e intervalos de confianza del tipo de cambio entre el peso uruguayo y el real brasileno.

**Variable principal:** uyu_por_brl, interpretada como pesos uruguayos necesarios para comprar un real brasileno.

## 1. Instalar dependencias

En Colab, la mayoria de las librerias ya estan instaladas. Esta celda asegura que esten disponibles las necesarias.

In [ ]:
!pip -q install pandas numpy matplotlib scipy openpyxl

## 2. Cargar el dataset

Opcion recomendada: leer el CSV desde GitHub. Si el archivo cambia de rama o nombre, modificar DATA_URL. Tambien se puede subir manualmente el CSV a Colab y cambiar pd.read_csv(DATA_URL) por pd.read_csv('dataset_uyu_brl_2005_2026.csv').

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_URL = 'https://raw.githubusercontent.com/valentinzamit26/Proyecto-PYE/main/dataset_uyu_brl_2005_2026.csv'
df = pd.read_csv(DATA_URL)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

df.head()

## 3. Validaciones basicas

Se valida que la variable principal y su inversa no esten confundidas. Debe cumplirse aproximadamente uyu_por_brl * brl_por_uyu = 1.

In [ ]:
expected = ['fecha', 'brl_por_usd', 'uyu_por_usd', 'brl_por_uyu', 'uyu_por_brl']
missing = [col for col in expected if col not in df.columns]
if missing:
    raise ValueError(f'Faltan columnas: {missing}')

validaciones = pd.DataFrame({
    'indicador': ['filas', 'columnas', 'nulos_totales', 'duplicados_fecha', 'max_error_producto_inverso'],
    'valor': [
        len(df),
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df['fecha'].duplicated().sum()),
        float((df['uyu_por_brl'] * df['brl_por_uyu'] - 1).abs().max()),
    ]
})
validaciones

## 4. Estadistica descriptiva

Resumen de la variable principal uyu_por_brl. La moda puede no ser muy informativa porque los datos son continuos y tienen muchos valores distintos, pero se incluye porque fue solicitada en la revision.

In [ ]:
x = df['uyu_por_brl'].dropna()
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
mode_counts = x.value_counts()
max_mode_count = int(mode_counts.max())
modes = sorted(mode_counts[mode_counts == max_mode_count].index.tolist())

descriptivo = pd.DataFrame({
    'medida': ['n', 'media', 'mediana', 'moda_1', 'moda_2', 'frecuencia_moda', 'q1', 'q3', 'iqr', 'varianza_muestral', 'desvio_estandar_muestral', 'minimo', 'maximo', 'rango', 'asimetria', 'curtosis', 'outliers_iqr'],
    'valor': [len(x), x.mean(), x.median(), modes[0], modes[1] if len(modes) > 1 else modes[0], max_mode_count, q1, q3, iqr, x.var(ddof=1), x.std(ddof=1), x.min(), x.max(), x.max() - x.min(), x.skew(), x.kurt(), int(((x < lower) | (x > upper)).sum())]
})
descriptivo

## 5. Graficos principales

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['fecha'], df['uyu_por_brl'], linewidth=1)
plt.title('Evolucion diaria del tipo de cambio UYU/BRL')
plt.xlabel('Fecha')
plt.ylabel('UYU por 1 BRL')
plt.grid(alpha=0.3)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x, bins=40, edgecolor='white')
axes[0].set_title('Histograma de UYU/BRL')
axes[0].set_xlabel('UYU por 1 BRL')
axes[1].boxplot(x, vert=False, patch_artist=True)
axes[1].set_title('Boxplot de UYU/BRL')
axes[1].set_xlabel('UYU por 1 BRL')
plt.tight_layout()
plt.show()

stats.probplot(x, dist='norm', plot=plt)
plt.title('Q-Q plot de UYU/BRL frente a Normal')
plt.show()

## 6. Autocorrelacion de la cotizacion

Como la variable es una serie temporal, se revisan rezagos antes de interpretar intervalos clasicos como si todas las observaciones fueran independientes.

In [ ]:
lag_df = pd.DataFrame({'x_t': x})
lag_df['x_t_1'] = lag_df['x_t'].shift(1)
lag_df['x_t_2'] = lag_df['x_t'].shift(2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(lag_df['x_t_1'], lag_df['x_t'], s=10, alpha=0.45)
axes[0].set_title('X_t vs X_{t-1}')
axes[0].set_xlabel('UYU/BRL en t-1')
axes[0].set_ylabel('UYU/BRL en t')
axes[1].scatter(lag_df['x_t_2'], lag_df['x_t'], s=10, alpha=0.45)
axes[1].set_title('X_t vs X_{t-2}')
axes[1].set_xlabel('UYU/BRL en t-2')
axes[1].set_ylabel('UYU/BRL en t')
plt.tight_layout()
plt.show()

def autocorr(values, nlags=30):
    values = np.asarray(values) - np.mean(values)
    denom = np.dot(values, values)
    return [1.0] + [float(np.dot(values[:-lag], values[lag:]) / denom) for lag in range(1, nlags + 1)]

lags = np.arange(31)
acf_values = autocorr(x, 30)
plt.figure(figsize=(10, 4))
plt.bar(lags, acf_values)
plt.title('Autocorrelacion de UYU/BRL')
plt.xlabel('Rezago')
plt.ylabel('Autocorrelacion')
plt.ylim(0, 1.05)
plt.show()

pd.DataFrame({'lag': [1, 2, 5, 10, 20, 30], 'autocorrelacion': [acf_values[i] for i in [1, 2, 5, 10, 20, 30]]})

## 7. Bondad de ajuste chi-cuadrado

Se comparan distribuciones continuas compatibles con el curso y con la consigna: Normal, Log-normal y Gamma. La mejor se elige en sentido relativo por menor estadistico chi-cuadrado.

In [ ]:
def chi_square_fit(values, bins=12):
    candidates = [
        ('Normal', stats.norm, stats.norm.fit(values), 2),
        ('Log-normal', stats.lognorm, stats.lognorm.fit(values, floc=0), 3),
        ('Gamma', stats.gamma, stats.gamma.fit(values, floc=0), 3),
    ]
    rows = []
    class_tables = {}
    n = len(values)
    for name, dist, params, estimated_params in candidates:
        edges = dist.ppf(np.linspace(0, 1, bins + 1), *params)
        edges[0], edges[-1] = -np.inf, np.inf
        observed, _ = np.histogram(values, bins=edges)
        expected = np.full(bins, n / bins)
        chi2 = float(((observed - expected) ** 2 / expected).sum())
        dof = bins - 1 - estimated_params
        p_value = stats.chi2.sf(chi2, dof)
        rows.append({'distribucion': name, 'chi2_estadistico': chi2, 'grados_libertad': dof, 'p_value': p_value, 'decision_5%': 'Se rechaza H0' if p_value < 0.05 else 'No se rechaza H0'})
        class_tables[name] = pd.DataFrame({'clase': range(1, bins + 1), 'observada': observed, 'esperada': expected, 'aporte_chi2': (observed - expected) ** 2 / expected})
    return pd.DataFrame(rows).sort_values('chi2_estadistico'), class_tables

fits, clases = chi_square_fit(x)
fits

In [ ]:
clases['Log-normal']

## 8. Intervalos de confianza

Se construyen intervalos clasicos para la media y la varianza. En la interpretacion final hay que recordar que la autocorrelacion alta reduce la independencia efectiva de la muestra.

In [ ]:
rows_media = []
rows_varianza = []
n = len(x)
mean = x.mean()
se = x.std(ddof=1) / np.sqrt(n)
s2 = x.var(ddof=1)
for level in [0.90, 0.95, 0.99]:
    alpha = 1 - level
    crit_t = stats.t.ppf(1 - alpha / 2, df=n - 1)
    rows_media.append({'nivel_confianza': level, 'media': mean, 'limite_inferior': mean - crit_t * se, 'limite_superior': mean + crit_t * se, 'amplitud': 2 * crit_t * se})
    li_var = (n - 1) * s2 / stats.chi2.ppf(1 - alpha / 2, df=n - 1)
    ls_var = (n - 1) * s2 / stats.chi2.ppf(alpha / 2, df=n - 1)
    rows_varianza.append({'nivel_confianza': level, 'varianza_muestral': s2, 'limite_inferior': li_var, 'limite_superior': ls_var, 'amplitud': ls_var - li_var})

pd.DataFrame(rows_media), pd.DataFrame(rows_varianza)

## 9. Nota metodologica

Python se usa como herramienta computacional para calcular tablas y graficos; no es un metodo estadistico. Una correlacion, autocorrelacion o relacion algebraica no demuestra causalidad economica. Ademas, al tratarse de una serie temporal con autocorrelacion alta, los intervalos clasicos deben interpretarse como aproximaciones bajo supuestos que no se cumplen plenamente.